In [ ]:
# !pip install scikit-learn

In [ ]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

In [ ]:
# import all libraries needed downstream
import os
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
system_size = 30

In [ ]:
!nvidia-smi

In [ ]:
datakit_data = np.load(f'/home/oarowolo/workfile/OPFData/data/Datakit/full_topology/pglib_opf_case{system_size}_ieee.npz', allow_pickle=True)

# Get all keys
print("Available keys in the dataset:")
for key in datakit_data.files:
    # Print the key and its array shape
    print(f"{key}: shape {datakit_data[key].shape}")

In [ ]:
datakit_data_bus = np.stack(datakit_data['bus_data'])
datakit_data_edge = np.stack(datakit_data['edge_data'])
datakit_data_gen = np.stack(datakit_data['gen_data'])
datakit_edge_index = np.stack(datakit_data['edge_index'])

In [ ]:
def load_grid_data(grid_size:int):
   # Load the data
    data = np.load(f'/home/oarowolo/workfile/OPFData/data/OPFData/full_topology/{grid_size}bus_combined_dataset.npz')

    # Get all keys
    print("Available keys in the dataset:")
    for key in data.files:
        # Print the key and its array shape
        print(f"{key}: shape {data[key].shape}") 
        
        
        # Access grid input features
        grid_bus = data['grid_bus'] 
        grid_generator = data['grid_generator']
        grid_load = data['grid_load']
        grid_shunt = data['grid_shunt']
        grid_ac_line_features = data['grid_ac_line_features']
        grid_transformer_features = data['grid_transformer_features']
        grid_ac_line_receivers = data['grid_ac_line_receivers']
        grid_ac_line_senders = data['grid_ac_line_senders']
        grid_transformer_senders = data['grid_transformer_senders']
        grid_transformer_receivers = data['grid_transformer_receivers']
        grid_generator_link_receivers = data['grid_generator_link_receivers']
        
        branch_list = list(zip(grid_ac_line_senders[0], grid_ac_line_receivers[0]))
        transformer_list = list(zip(grid_transformer_senders[0], grid_transformer_receivers[0]))
        for k in transformer_list:
            branch_list.append(k)
            
        grid_load_link_receivers = data['grid_load_link_receivers']
        grid_shunt_link_receivers = data['grid_shunt_link_receivers']
        
        generator_indices = grid_generator_link_receivers[0]
        load_indices = grid_load_link_receivers[0]
        shunt_indices = grid_shunt_link_receivers[0]
        
        solution_bus = data['solution_bus']  
        solution_generator = data['solution_generator'] 
        solution_objective = data['metadata_objective']
        
        
        return grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features, grid_transformer_features, solution_bus, solution_generator, solution_objective, branch_list, generator_indices,load_indices,shunt_indices

In [ ]:
grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features, grid_transformer_features, solution_bus, solution_generator, solution_objective, branch_list, generator_indices,load_indices,shunt_indices = load_grid_data(system_size)

In [ ]:
## We have to use the load inputs, gen and bus outputs,variable gencosts, and rx from the Datakit data
## Keep the rest of the system features from OPFData since it's literally the same system
## We would use the rest of the system features to compute constraint satisfaction and so on.
## note that datakit does not automatically save the objective cost

In [ ]:
datakit_load = datakit_data_bus[:,:,:2]
datakit_solution_generator = datakit_data_bus[:,:,2:4]
datakit_solution_bus = datakit_data_bus[:,:,4:]
datakit_r_x = datakit_data_edge[:,:,:2]
datakit_gencosts = datakit_data_gen[:,:,:3]

In [ ]:
datakit_solution_objectives = datakit_gencosts[:,:,2] * (datakit_solution_generator[:,generator_indices,0]**2) + datakit_gencosts[:,:,1] * (datakit_solution_generator[:,generator_indices,0]) + datakit_gencosts[:,:,2]
datakit_solution_objectives = datakit_solution_objectives.sum(axis=1) * 100 ##multiplying by the baseMVA because the costs are in per MW

In [ ]:
#what we want to do is separate the datakit_rx data into lines and transformers
#we can know the number of transformers from the second index of transformer features x and select the last x branches in branch list
# use the transformer branch tuples to find their indices in the datakit edge index data, use the indices to replace the original r_x in the OPF data
transformer_branches = branch_list[-grid_transformer_features.shape[1]:]
# Compare all edges at once
datakit_edge_list = datakit_edge_index[0]
transformer_branches = np.array(transformer_branches)
matches = (datakit_edge_list[0][:, None] == transformer_branches[:, 0]) & (datakit_edge_list[1][:, None] == transformer_branches[:, 1])
transformer_indices = np.where(matches.any(axis=1))[0]
line_indices = np.where(matches.any(axis=1) == False)[0]

In [ ]:
line_rx = datakit_r_x[:,line_indices,:]
transformer_rx = datakit_r_x[:,transformer_indices,:]

In [ ]:
grid_load = datakit_load[:,load_indices,:]
solution_bus = datakit_solution_bus  
solution_bus = solution_bus[:, :, ::-1]  # reverse the order of Va and Vm to match OPFData
solution_bus[:,:,0] = np.radians(solution_bus[:,:,0]) ## we also need to convert angle to radians for consistency
solution_generator = datakit_solution_generator[:,generator_indices,:]
solution_objective = datakit_solution_objectives

In [ ]:
class FF_DNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,num_layers):
        super(FF_DNN, self).__init__()
        self.num_layers = num_layers
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_dim=hidden_dim

        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())

        for _ in range(num_layers-1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_dim, output_dim))
        
        layers.append(nn.Sigmoid())

        self.model = nn.Sequential(*layers)

    def forward(self, x):

        out = self.model(x)
        return out   

In [ ]:
def train_model(model, loader,optimizer, criterion):
#     model.to(device)
    model.train()

    epoch_loss = 0

    for inputs, targets in loader:
        
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        batch_size = inputs.size(0)
        outputs = model(inputs)

        b_up, b_down = convert_bounds(inputs,outputs)
        b_up = b_up.to(device)
        b_down = b_down.to(device)       
        outputs = outputs * (b_up - b_down) + b_down
        outputs = torch.clamp(outputs,min=b_down, max=b_up)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss/len(loader)

In [ ]:
def evaluate_model(model, loader, criterion):
#     model.to(device)
    model.eval() # specifies that the model is in evaluation mode

    epoch_loss = 0

    with torch.no_grad():

        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            batch_size = inputs.size(0)
            outputs = model(inputs)

            b_up, b_down = convert_bounds(inputs,outputs)
            b_up = b_up.to(device)
            b_down = b_down.to(device)

            outputs = outputs * (b_up - b_down) + b_down
            outputs = torch.clamp(outputs,min=b_down, max=b_up)
            
            loss = criterion(outputs, targets)

            epoch_loss += loss.item()

        return epoch_loss / len(loader)

In [ ]:
def test_model(model, loader):
#     model.to(device)
    model.eval() # specifies that the model is in evaluation mode

    epoch_loss = 0
    model_preds = []
    actual_y = []

    with torch.no_grad():

        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            batch_size = inputs.size(0)
            outputs = model(inputs)

            b_up, b_down = convert_bounds(inputs,outputs)
            b_up = b_up.to(device)
            b_down = b_down.to(device)
            outputs = outputs * (b_up - b_down) + b_down
            outputs = torch.clamp(outputs,min=b_down, max=b_up)
            
            model_preds.append(outputs)
            loss = criterion(outputs, targets)
            actual_y.append(targets)
            epoch_loss += loss.item()

        return epoch_loss / len(loader), model_preds, actual_y

In [ ]:
# generator_costs = torch.tensor(grid_generator[0,:,8:11]).float().to(device)
system_objectives = torch.tensor(solution_objective).float().to(device)

In [ ]:
# for features we know are constant, we can simply choose the first index and multiply by the first dimension of Datakit data 

In [ ]:
samp_grid_bus = grid_bus[0]
samp_grid_generator = grid_generator[0]
samp_grid_shunt = grid_shunt[0]
samp_grid_ac_line_features = grid_ac_line_features[0]
samp_grid_transformer_features = grid_transformer_features[0]

In [ ]:
grid_bus = np.repeat(samp_grid_bus[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_generator = np.repeat(samp_grid_generator[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_shunt = np.repeat(samp_grid_shunt[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_ac_line_features = np.repeat(samp_grid_ac_line_features[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_transformer_features = np.repeat(samp_grid_transformer_features[np.newaxis, :, :], grid_load.shape[0], axis=0)

In [ ]:
## grid AC-line/transformer features are not constant, we should not act like they are!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
# now we replace the fixed generator costs and the fix rx with the actual variable costs and variable rx

In [ ]:
grid_generator[:,:,8:] = datakit_gencosts
grid_ac_line_features[:,:,4:6] = line_rx
grid_transformer_features[:,:,2:4] = transformer_rx

In [ ]:
grid_bus_0 = grid_bus
grid_generator_0 = grid_generator
grid_shunt_0 = grid_shunt
grid_load_0 = grid_load
grid_ac_line_features_0 = grid_ac_line_features
grid_transformer_features_0 = grid_transformer_features

In [ ]:
gen_outputs_0 = solution_generator[0,:,:]

In [ ]:
#create grid inputs 
grid_bus = grid_bus.reshape(grid_bus.shape[0],-1)
grid_generator = grid_generator.reshape(grid_generator.shape[0],-1)
grid_load = grid_load.reshape(grid_load.shape[0],-1)
grid_shunt = grid_shunt.reshape(grid_shunt.shape[0],-1)
grid_ac_line_features = grid_ac_line_features.reshape(grid_ac_line_features.shape[0],-1)
grid_transformer_features = grid_transformer_features.reshape(grid_transformer_features.shape[0],-1)

grid_input = np.concatenate((grid_bus,grid_generator,grid_load,grid_shunt,grid_ac_line_features,grid_transformer_features),axis=1)

In [ ]:
#create grid outputs
solution_bus = solution_bus.reshape(solution_bus.shape[0],-1)  # Shape: (num_files, num_bus_nodes)
solution_generator = solution_generator.reshape(solution_generator.shape[0],-1)

grid_output = np.concatenate((solution_bus,solution_generator),axis=1)

In [ ]:
grid_input = torch.tensor(grid_input).float()
grid_output = torch.tensor(grid_output).float()

In [ ]:
def convert_bounds(model_input,model_output):

    single_model_input = model_input[0]

    batch_size = model_output.shape[0]
    out_size = model_output.shape[-1]

    vmin = single_model_input[:grid_bus.shape[-1]][2::4]

    vmax = single_model_input[:grid_bus.shape[-1]][3::4]

    pmin = single_model_input[grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]][2::11]

    pmax = single_model_input[grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]][3::11]

    
    qmin = single_model_input[grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]][5::11]

    qmax = single_model_input[grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]][6::11]

    thetamin =  torch.tensor([-2.0])
    thetamin = thetamin.tile((grid_bus_0.shape[1],))

    thetamax =  torch.tensor([2.0])
    thetamax = thetamax.tile((grid_bus_0.shape[1],))

    bounds_up = torch.zeros(out_size,)
    
    bounds_up[0:solution_bus.shape[-1]:2] = thetamax
    bounds_up[1:solution_bus.shape[-1]:2] = vmax
    bounds_up[solution_bus.shape[-1]::2] = pmax
    bounds_up[solution_bus.shape[-1]+1::2] = qmax

    bounds_down = torch.zeros(out_size,)
    bounds_down[:solution_bus.shape[-1]:2] = thetamin
    bounds_down[1:solution_bus.shape[-1]:2] = vmin
    bounds_down[solution_bus.shape[-1]::2] = pmin
    bounds_down[solution_bus.shape[-1]+1::2] = qmin

    bounds_up =  bounds_up.tile((batch_size,1))
    bounds_down = bounds_down.tile((batch_size,1))

    return bounds_up, bounds_down    

In [ ]:
def train_val_test_split(tensor_input, tensor_output, system_objective, system_gencosts, train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = tensor_input.shape[0]
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Split tensors
    tensor_input_train, tensor_output_train = tensor_input[train_indices], tensor_output[train_indices]
    tensor_input_val, tensor_output_val = tensor_input[val_indices], tensor_output[val_indices]
    tensor_input_test, tensor_output_test = tensor_input[test_indices], tensor_output[test_indices]

    test_objectives = system_objective[test_indices]
    test_gen_costs = system_gencosts[test_indices]

    return (tensor_input_train, tensor_input_val, tensor_input_test), (tensor_output_train, tensor_output_val, tensor_output_test), (test_objectives, test_gen_costs)

In [ ]:
tensor_inputs, tensor_outputs, test_obj_costs = train_val_test_split(grid_input, grid_output, system_objectives,datakit_gencosts )

In [ ]:
train_x, val_x, test_x = tensor_inputs
train_y, val_y, test_y = tensor_outputs
test_obj, test_cost = test_obj_costs
test_cost = torch.tensor(test_cost).to(device)

In [ ]:
train_dataset = TensorDataset(train_x, train_y)
val_dataset = TensorDataset(val_x, val_y)
test_dataset = TensorDataset(test_x, test_y)

In [ ]:
## wandb set-up
api_key = 'my_key'
wandb.login(key=api_key)

In [ ]:
input_dim = train_x.shape[1]
output_dim = train_y.shape[1]

hidden_dim = 256 
num_layers = 5
batch_size = 256

model = FF_DNN(input_dim,hidden_dim,output_dim,num_layers).to(device)

In [ ]:
def compute_optimality(model_outputs, gen_costs, objective):

    c2 = gen_costs[:,:,2] * 100 ## multiply costs by 100 to account for the per MW costs of Datakit
    c1 = gen_costs[:,:,1] * 100
    c0 = gen_costs[:,:,0] * 100

    pg_starting_index = model_outputs.shape[-1] - (gen_costs.shape[1]*2)

    # Get relevant output dimensions (zero-indexed)
    out_dim2 = model_outputs[:, pg_starting_index:] # select on Pgs for generators
    out_dim2 = out_dim2[:,::2]  

    print('the shape of out dim is ', out_dim2.shape)
    print('the shape of c1 is ', c1.shape)

    # we don't need to tile costs again because costs are now different
    
    # c0 = c0.unsqueeze(0)
    # c1 = c1.unsqueeze(0)
    # c2 = c2.unsqueeze(0)

    # c0 = c0.repeat(out_dim2.shape[0],1) # tile cost tensors since the cost of each generator stays the same from sample to sample
    # c1 = c1.repeat(out_dim2.shape[0],1)
    # c2 = c2.repeat(out_dim2.shape[0],1)
    
    # Compute node-wise metrics
    system_metrics = c2 * (out_dim2 ** 2) + c1 * out_dim2 + c0

    model_obj = torch.sum(system_metrics, dim=1)

    print(f'average model objective is {model_obj.mean()}')
    print(f'average IPOPT objective is {objective.mean()}')

    optimality_gap = (model_obj / objective) * 100

    
    return optimality_gap.mean()

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"Datakit_{system_size}_FNN_5_256_PQVT",
      # Track hyperparameters and run metadata
      config={
      "architecture": "FNN",
      "dataset": "Datakit",
      "epochs": 100,
      })

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5,weight_decay=5e-8)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
training_losses = []
validation_losses = []
best_valid_loss = float('inf')
early_stop_thresh = 20
best_epoch = -1
best_model_state = None
num_epochs= 100


start_training_time = time.time()
for epoch in tqdm(range(num_epochs), desc="Training Progress"):
    train_loss = train_model(model, train_loader, optimizer,criterion)
    valid_loss = evaluate_model(model, val_loader,criterion)
    training_losses.append(train_loss)
    validation_losses.append(valid_loss)

    wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

    scheduler.step(valid_loss)

    if epoch % 10 == 0:
      print(f'Epoch: {epoch}')
      print(f'\tTrain Loss: {train_loss:.4f}')
      print(f'\t Val. Loss: {valid_loss:.4f}')
    if valid_loss < best_valid_loss:
      best_valid_loss = valid_loss
      best_model_state = deepcopy(model.state_dict())

end_training_time = time.time()
total_training_time = end_training_time - start_training_time
print(f"Training time taken is: {total_training_time} seconds")

plt.subplots(figsize=(5,3))
plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
plt.legend()
plt.title(f'FNN Training and Validation loss for {system_size} Bus system',fontsize = 15)
plt.xlabel('Epochs',fontsize = 12)
plt.ylabel('MSE Loss',fontsize = 12)
plt.semilogy()

training_losses=np.array(training_losses)
validation_losses=np.array(validation_losses)


model.load_state_dict(best_model_state)
model.eval()

In [ ]:
torch.save(model.state_dict(), f"{system_size}_bus_FNN_5_256_PQVT.pth")
wandb.save(f"{system_size}_bus_FNN_5_256_PQVT.pth")  # Upload to WandB

In [ ]:
test_loss, predicted_output_list, grid_output_list = test_model(model, test_loader)

In [ ]:
print('test loss is ', test_loss)

In [ ]:
predicted_output = torch.cat(predicted_output_list, dim=0)

In [ ]:
grid_output = torch.cat(grid_output_list, dim=0)

In [ ]:
# calculate the optimality gap to compare with other methods
average_test_opt_gap = compute_optimality(predicted_output,test_cost, test_obj)

In [ ]:
print(f'The average optimality gap on test data is: {average_test_opt_gap:.5f}')

In [ ]:
def calculate_angle_differences(angles, edges):
    
    # Initialize array to store angle differences
    angle_differences = torch.zeros(len(edges), dtype=torch.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,:,4:5]
    line_x = edge_inputs[:,:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
voltage_predictions = predicted_output[:,:system_size*2]
voltage_targets = grid_output[:,:system_size*2]

In [ ]:
voltage_predictions = voltage_predictions.reshape(-1,system_size,2).to('cpu')
voltage_targets = voltage_targets.reshape(-1,system_size,2).to('cpu')

In [ ]:
calc_loss = nn.MSELoss()
voltage_angle_loss = calc_loss(voltage_predictions[:,:,0],voltage_targets[:,:,0])
voltage_magnitude_loss = calc_loss(voltage_predictions[:,:,1],voltage_targets[:,:,1])
print('average voltage angle discrepancy is  ', voltage_angle_loss)
print('average voltage magnitude discrepancy is  ', voltage_magnitude_loss)

In [ ]:
load_start_index = grid_bus.shape[-1]+grid_generator.shape[-1]
load_end_index = load_start_index+grid_load.shape[-1]
test_load = test_x[:,load_start_index:load_end_index]
test_load = test_load.reshape(-1,grid_load_0.shape[1],grid_load_0.shape[2])

In [ ]:
shunt_start_index = load_end_index
shunt_end_index = shunt_start_index + grid_shunt.shape[-1]
test_shunt = test_x[:,shunt_start_index:shunt_end_index]
test_shunt = test_shunt.reshape(-1,grid_shunt_0.shape[1],grid_shunt_0.shape[2])

In [ ]:
lines_start_index = shunt_end_index
lines_end_index = lines_start_index + grid_ac_line_features.shape[-1]
test_lines = test_x[:,lines_start_index:lines_end_index]
test_lines = test_lines.reshape(-1,grid_ac_line_features_0.shape[1],grid_ac_line_features_0.shape[2])

In [ ]:
transformer_start_index = lines_end_index
transformer_end_index = transformer_start_index + grid_transformer_features.shape[-1]
test_transformers = test_x[:,transformer_start_index:transformer_end_index]
test_transformers = test_transformers.reshape(-1,grid_transformer_features_0.shape[1],grid_transformer_features_0.shape[2])

In [ ]:
edge_inputs = torch.zeros((test_x.shape[0],len(branch_list),11))

In [ ]:
edge_inputs[:,:grid_ac_line_features_0.shape[1],:9] = test_lines  # rearranging edge inputs to align for transformers and transmission lines
edge_inputs[:,grid_ac_line_features_0.shape[1]:,:2] =  test_transformers[:,:,:2]
edge_inputs[:,grid_ac_line_features_0.shape[1]:,2:4] =  test_transformers[:,:,9:]
edge_inputs[:,grid_ac_line_features_0.shape[1]:,4:9] =  test_transformers[:,:,2:7]
edge_inputs[:,grid_ac_line_features_0.shape[1]:,9:] =  test_transformers[:,:,7:9]
edge_inputs[:,:grid_ac_line_features_0.shape[1],9:10] = 1.0

In [ ]:
edge_g, edge_b = compute_gandb(edge_inputs)

In [ ]:
def convert_to_complex_voltage(voltage_tensor):
    # Extract angle and magnitude
    voltage_angle = voltage_tensor[:,:,0:1]  # In radians
    voltage_magnitude = voltage_tensor[:,:,1:]
    
    # Calculate real and imaginary parts
    real_voltage = voltage_magnitude * torch.cos(voltage_angle)
    imaginary_voltage = voltage_magnitude * torch.sin(voltage_angle)
    
    return torch.concat((real_voltage,imaginary_voltage),dim=2)

In [ ]:
# def convert_to_complex_rectangle(tensor_2d):
#     # Extract angle and magnitude
#     tensor_mag = tensor_2d[:,0:1]  # In radians
#     tensor_angle = tensor_2d[:,1:]
    
#     # Calculate real and imaginary parts
#     real_tensor = tensor_mag * torch.cos(tensor_angle)
#     imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
#     return torch.concat((real_tensor,imaginary_tensor),dim=1)

In [ ]:
def convert_to_complex_rectangle_3D(tensor_3d):
    # Extract angle and magnitude
    tensor_mag = tensor_3d[:,:,0:1]  # In radians
    tensor_angle = tensor_3d[:,:,1:]
    
    # Calculate real and imaginary parts
    real_tensor = tensor_mag * torch.cos(tensor_angle)
    imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
    return torch.concat((real_tensor,imaginary_tensor),dim=2)

In [ ]:
def calculate_only_branch_flows(
    demand: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
    voltage: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
    branches: list,  # list of 20 tuples (from_bus, to_bus)
    Yks: torch.Tensor,  # shape (batch_size,14,2) [real, imag] shunt admittance
    Yij: torch.Tensor,  # shape (batch_size,20,2) [real, imag] branch admittance
    Yijc: torch.Tensor,  # shape (batch_size,20,2) [real, imag] branch charging admittance
    Tij: torch.Tensor,  # shape (batch_size,20,2) [real, imag] transformation ratio
) -> torch.Tensor:
    batch_size = demand.shape[0]
    num_nodes = voltage.shape[1]
    
    # Helper function for batched complex multiplication
    def complex_mult_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.stack([
            a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1],
            a[..., 0] * b[..., 1] + a[..., 1] * b[..., 0]
        ], dim=-1)

    # Helper function for batched complex conjugate
    def complex_conj_batch(x: torch.Tensor) -> torch.Tensor:
        return torch.stack([x[..., 0], -x[..., 1]], dim=-1)

    # Helper function for batched complex division
    def complex_div_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        denominator = b[..., 0]**2 + b[..., 1]**2
        return torch.stack([
            (a[..., 0] * b[..., 0] + a[..., 1] * b[..., 1]) / denominator,
            (a[..., 1] * b[..., 0] - a[..., 0] * b[..., 1]) / denominator
        ], dim=-1)

    # Initialize generator power tensor
    generator_power = torch.zeros_like(demand)
    
    # Calculate shunt power terms for each node (vectorized)
    v_mag_sq = torch.sum(voltage**2, dim=-1, keepdim=True)  # shape: (batch_size, 14, 1)
    v_mag_sq = torch.cat([v_mag_sq, torch.zeros_like(v_mag_sq)], dim=-1)  # shape: (batch_size, 14, 2)
    
    # Yks is already batched
    shunt_power = complex_mult_batch(complex_conj_batch(Yks), v_mag_sq)
    
    # Pre-allocate branch flows dictionary with tensors
    branch_flows = {}
    
    # Calculate branch flows (vectorized)
    for idx, (i, j) in enumerate(branches):
        
        # Get complex voltage at both ends
        vi = voltage[:, i]  # shape: (batch_size, 2)
        vj = voltage[:, j]  # shape: (batch_size, 2)
        
        # First term calculations
        vi_mag_sq = torch.sum(vi**2, dim=-1, keepdim=True)  # shape: (batch_size, 1)
        vi_mag_sq = torch.cat([vi_mag_sq, torch.zeros_like(vi_mag_sq)], dim=-1)  # shape: (batch_size, 2)
        
        tij_mag_sq = torch.sum(Tij[:, idx]**2, dim=-1, keepdim=True)  # shape: (batch_size, 1)
        tij_mag_sq_tensor = torch.cat([tij_mag_sq, torch.zeros_like(tij_mag_sq)], dim=-1)  # shape: (batch_size, 2)
        
        vi_over_tij_sq = complex_div_batch(vi_mag_sq, tij_mag_sq_tensor)
        
        # Sum of branch admittance and charging admittance (already batched)
        Y_total = Yij[:, idx] + Yijc[:, idx]  # shape: (batch_size, 2)
        
        term1 = complex_mult_batch(complex_conj_batch(Y_total), vi_over_tij_sq)
        
        # Second term calculations
        vivj = complex_mult_batch(vi, complex_conj_batch(vj))
        term2 = complex_mult_batch(
            complex_conj_batch(Yij[:, idx]),
            complex_div_batch(vivj, Tij[:, idx])
        )
        
        # Total branch flow Sij
        Sij = term1 - term2
        branch_flows[(i, j, idx)] = Sij
        
        # Reverse flow calculations
        vj_mag_sq = torch.sum(vj**2, dim=-1, keepdim=True)
        vj_mag_sq = torch.cat([vj_mag_sq, torch.zeros_like(vj_mag_sq)], dim=-1)
        
        term1_ji = complex_mult_batch(complex_conj_batch(Y_total), vj_mag_sq)
        vjvi = complex_mult_batch(complex_conj_batch(vi), vj)
        term2_ji = complex_mult_batch(
            complex_conj_batch(Yij[:, idx]),
            complex_div_batch(vjvi, complex_conj_batch(Tij[:, idx]))
        )
        
        Sji = term1_ji - term2_ji
        branch_flows[(j, i, idx)] = Sji
    
    
    # Aggregate generator power for each node (vectorized)
    for i in range(num_nodes):
        for index, (from_bus, to_bus) in enumerate(branches):
            if from_bus == i:
                generator_power[:, i] += branch_flows[(from_bus, to_bus, index)]
            if to_bus == i:
                generator_power[:, i] += branch_flows[(to_bus, from_bus, index)]
    
    generator_power += demand + shunt_power
    
    return generator_power, branch_flows

In [ ]:
load_input = torch.zeros((test_x.shape[0],system_size,2)).to(torch.float32)
load_input[:,load_indices,:] = test_load.to('cpu')

In [ ]:
shunt_input = torch.zeros((test_x.shape[0],system_size,2)).to(torch.float32)
shunt_input[:,shunt_indices,:] = test_shunt.to('cpu')
shunt_input = shunt_input[:,:, [1, 0]]

In [ ]:
conductance_susceptance = torch.cat((edge_g, edge_b), dim=2)
conductance_susceptance = conductance_susceptance.to('cpu').to(torch.float32)
charging_susceptance = torch.zeros_like(conductance_susceptance)
charging_susceptance[:,:,1:] =  edge_inputs[:,:,2:3].to('cpu').to(torch.float32)

In [ ]:
Tij = edge_inputs[:,:,9:].to('cpu')
Tij[:,:grid_ac_line_features_0.shape[1],0:1] = 1.0
Tij_rec = convert_to_complex_rectangle_3D(Tij)
Tij_rec = Tij_rec.to(torch.float32)

In [ ]:
complex_v = convert_to_complex_voltage(voltage_predictions) ### check if actual answer has no violation as a sanity check
complex_v = complex_v.to('cpu').to(torch.float32)

In [ ]:
injection_balance,branch_flows = calculate_only_branch_flows(load_input,complex_v,branch_list,shunt_input,conductance_susceptance,charging_susceptance,Tij_rec)

In [ ]:
##### time to evaluate constraint satisfactions

In [ ]:
predicted_angle_differences = torch.zeros((voltage_predictions.shape[0], len(branch_list)))
predicted_angles = voltage_predictions[:,:,0]

for j in range(voltage_predictions.shape[0]):
    angle_differences = calculate_angle_differences(predicted_angles[j],branch_list)
    predicted_angle_differences[j] = angle_differences

In [ ]:
true_angle_differences = torch.zeros((voltage_targets.shape[0], len(branch_list)))
true_angles = voltage_targets[:,:,0]

for j in range(voltage_targets.shape[0]):
    angle_differences = calculate_angle_differences(true_angles[j],branch_list)
    true_angle_differences[j] = angle_differences

In [ ]:
# voltage angle difference bound 
angle_diff_upper = torch.full(predicted_angle_differences.shape, 0.5236)
angle_diff_lower = torch.full(predicted_angle_differences.shape, -0.5236)

# Calculate violations
lower_angle_violations = torch.clamp(angle_diff_lower - predicted_angle_differences, min=0)  # Positive if below lower bound
upper_angle_violations = torch.clamp(predicted_angle_differences - angle_diff_upper, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
angle_diff_violations = lower_angle_violations + upper_angle_violations

print('max voltage angle difference violation is : ',angle_diff_violations.max() )
print('average voltage angle difference violation is : ',angle_diff_violations.mean())

In [ ]:
## voltage magnitude bound

vmin = test_x[:,:grid_bus.shape[-1]]
vmin = vmin[:,2::4].to('cpu')

vmax = test_x[:,:grid_bus.shape[-1]]
vmax = vmax[:,3::4].to('cpu')

lower_vmag_violation = torch.clamp(vmin - voltage_predictions[:,:,1:].squeeze(), min=0)  # Positive if below lower bound

upper_vmag_violation = torch.clamp(voltage_predictions[:,:,1:].squeeze() - vmax, min=0)  # Positive if above upper bound

vmag_violations = lower_vmag_violation + upper_vmag_violation

print('max voltage magnitude violation is : ',vmag_violations.max() )
print('average voltage magnitude violation is : ',vmag_violations.mean() )

In [ ]:
# Gen reactive power bounds 
qmin = test_x[:,grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]]
qmin = qmin[:,5::11].to('cpu')

qmax = test_x[:,grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]]
qmax = qmax[:,6::11].to('cpu')

gen_outputs = predicted_output[:,system_size*2:]
qgens= gen_outputs[:,1::2].to('cpu')

lower_qgen_violations = torch.clamp(qmin - qgens, min=0)  # Positive if below lower bound
upper_qgen_violations = torch.clamp(qgens - qmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
qgen_violations = lower_qgen_violations + upper_qgen_violations

print('max gen reactive power violation is : ',qgen_violations.max() )
print('average reactive power violation is : ',qgen_violations.mean() )

In [ ]:
# Gen active power bounds 
pmin = test_x[:,grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]]
pmin = pmin[:,2::11].to('cpu')

pmax = test_x[:,grid_bus.shape[-1]:grid_bus.shape[-1]+grid_generator.shape[-1]]
pmax = pmax[:,3::11].to('cpu')

gen_outputs = predicted_output[:,system_size*2:]
pgens= gen_outputs[:,::2].to('cpu')

lower_pgen_violations = torch.clamp(pmin - pgens, min=0)  # Positive if below lower bound
upper_pgen_violations = torch.clamp(pgens - pmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
pgen_violations = lower_pgen_violations + upper_pgen_violations

print('max gen active power violation is : ',pgen_violations.max() )
print('average active power violation is : ',pgen_violations.mean() )

In [ ]:
power_predictions = torch.cat((pgens.unsqueeze(-1),qgens.unsqueeze(-1)),dim=2)

In [ ]:
actual_pgens = grid_output[:,system_size*2:]
actual_pgens= actual_pgens[:,::2].to('cpu')

In [ ]:
actual_qgens = grid_output[:,system_size*2:]
actual_qgens= actual_qgens[:,1::2].to('cpu')

In [ ]:
calc_loss = nn.MSELoss()
active_power_loss = calc_loss(pgens,actual_pgens)
reactive_power_loss = calc_loss(qgens,actual_qgens)
print('average gen active power error is  ', active_power_loss)
print('average gen reactive power error is  ', reactive_power_loss)

In [ ]:
# torch.save(voltage_predictions, f"{system_size}_FNN_pre_powerflow_voltages.pt")
# torch.save(power_predictions, f"{system_size}_FNN_pre_powerflow_powers.pt")

In [ ]:
forward_keys = [(i, j, index) for index, (i,j) in enumerate(branch_list)]
reverse_keys = [(j, i, index) for index, (i,j) in enumerate(branch_list)]

In [ ]:
forward_branch_flows = {key: branch_flows[key] for key in forward_keys if key in branch_flows}
reverse_branch_flows = {key: branch_flows[key] for key in reverse_keys if key in branch_flows}

In [ ]:
forward_power_flows = [tensor.unsqueeze(dim=1) for tensor in forward_branch_flows.values()]

# Step 2: Concatenate tensors along the second axis (dim=1)
forward_power_flows = torch.cat(forward_power_flows, dim=1)

In [ ]:
reverse_power_flows = [tensor.unsqueeze(dim=1) for tensor in reverse_branch_flows.values()]

# Step 2: Concatenate tensors along the second axis (dim=1)
reverse_power_flows = torch.cat(reverse_power_flows, dim=1)

In [ ]:
# compute the power magnitudes

# Separate real and imaginary parts
def convert_to_power_magnitude(power_flow):
    
    real = power_flow[..., 0]  
    imag = power_flow[..., 1]  

    # Compute the magnitudes
    magnitudes = torch.sqrt(real**2 + imag**2) 

    result_tensor = magnitudes.unsqueeze(-1)

    return result_tensor

In [ ]:
forward_flow_magnitude = convert_to_power_magnitude(forward_power_flows)
reverse_flow_magnitude = convert_to_power_magnitude(reverse_power_flows)


In [ ]:
# Branch flow bounds in forward direction

long_term_line_rating = edge_inputs[0][:,6:7].to('cpu')

branch_flow_limit = long_term_line_rating.tile((test_x.shape[0],1,1))

forward_branch_flow = forward_flow_magnitude

forward_flow_violations = torch.clamp(forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound

print('max forward power flow violation is : ',forward_flow_violations.max() )
print('average  forward power flow violation is : ',forward_flow_violations.mean())

In [ ]:
# Branch flow bounds in reverse direction

reverse_branch_flow = reverse_flow_magnitude

reverse_flow_violations = torch.clamp(reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound

print('max reverse power flow violation is : ', reverse_flow_violations.max() )
print('average reverse power flow violation is : ', reverse_flow_violations.mean() )

In [ ]:
generator_Ps = torch.zeros((test_x.shape[0],system_size,1))
# generator_Ps[:,generator_indices,:] = actual_pgens.unsqueeze(-1)
for i in range(test_x.shape[0]):
    generator_Ps[i, :, :].index_add_(0, torch.tensor(generator_indices), pgens.unsqueeze(-1)[i, :, :])

In [ ]:
generator_Qs = torch.zeros((test_x.shape[0],system_size,1))
# generator_Qs[:,generator_indices,:] = actual_qgens.unsqueeze(-1)

for i in range(test_x.shape[0]):
    generator_Qs[i, :, :].index_add_(0, torch.tensor(generator_indices), qgens.unsqueeze(-1)[i, :, :])

In [ ]:
## Evaluate power balance contraint violations

real_power_balance_mismatches = injection_balance[:,:,0:1] - generator_Ps

print('max active power balance mismatch is : ', real_power_balance_mismatches.max())
print('average active power balance mismatch is : ', real_power_balance_mismatches.mean())

In [ ]:
reactive_power_balance_mismatches = injection_balance[:,:,1:] - generator_Qs


print('max reactive power balance mismatch is : ', reactive_power_balance_mismatches.max())
print('average reactive power balance mismatch is : ', reactive_power_balance_mismatches.mean())

In [ ]:
#create table to save important metrics
columns =["metric", "value"]
model_metrics_table = wandb.Table(columns=columns)

In [ ]:
model_metrics_table.add_data("optimality gap", average_test_opt_gap)
model_metrics_table.add_data("max voltage angle difference violation", angle_diff_violations.max())
model_metrics_table.add_data("average voltage angle difference violation", angle_diff_violations.mean())
model_metrics_table.add_data("max voltage magnitude violation", vmag_violations.max())
model_metrics_table.add_data("average voltage magnitude violation", vmag_violations.mean())
model_metrics_table.add_data("max gen active power violation", pgen_violations.max())
model_metrics_table.add_data("average gen active power violation", pgen_violations.mean())
model_metrics_table.add_data("max gen reactive power violation", qgen_violations.max())
model_metrics_table.add_data("average gen reactive power violation", qgen_violations.mean())
model_metrics_table.add_data("max forward power flows violation", forward_flow_violations.max())
model_metrics_table.add_data("average forward power flows violation", forward_flow_violations.mean())
model_metrics_table.add_data("max reverse power flows violation", reverse_flow_violations.max())
model_metrics_table.add_data("average reverse power flows violation", reverse_flow_violations.mean())
model_metrics_table.add_data("max active power balance mismatch", real_power_balance_mismatches.max())
model_metrics_table.add_data("average active power balance mismatch", real_power_balance_mismatches.mean())
model_metrics_table.add_data("max reactive power balance mismatch", reactive_power_balance_mismatches.max())
model_metrics_table.add_data("average reactive power balance mismatch", reactive_power_balance_mismatches.mean())
wandb.log({"model_metrics_table" : model_metrics_table})

In [ ]:
wandb.finish()

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()